# Người cầm súng ngắn · handgun hand_01: pose → crop quanh tay → ResNet

Cùng cách làm với nhánh vật sắc nhọn (hand_04): YOLO **pose** tìm người và khuỷu tay / cổ tay, cắt ô vuông quanh
**mỗi bàn tay**, rồi **ResNet50** phân loại ô đó: tay *có* hay *không* cầm súng ngắn. Súng ngắn nhỏ và nằm gọn trong
lòng bàn tay, nên phóng to vùng tay lên 224 px giúp thấy rõ hơn hẳn so với detector chạy trên cả khung hình.

**Dữ liệu công khai** (tất cả qua `extensions/sharp_object/collect.py` với `WEAPON_CLASS_MAP=datasets/handgun/class_map.json`):

| Nguồn | Dương (súng) | Âm |
|---|---|---|
| Granada *Pistol detection* | 3000 ảnh, nhãn `pistol` | |
| Granada *Sohas* (tay cầm đồ vật) | `pistol` | tay cầm dao, điện thoại, ví, tiền, thẻ |
| Zenodo Dangerous Items | `gun` | ảnh chỉ có dao / mã tấu / gậy (ảnh có `rifle` bị bỏ) |
| Open Images | `Handgun` | điện thoại, remote, máy sấy, chuột, dập ghim, máy ảnh, đèn pin, máy tính bỏ túi |
| Simuletic ATM (giả lập CCTV) | `gun` | cảnh cướp bằng dao |
| COCO `coco_hold`, Charades `charades_neg` | | người cầm điện thoại / cốc / sách / chổi… trong nhà |

**Dữ liệu tự quay (không bắt buộc):** zip CVAT *YOLO 1.1* (hoặc thư mục Ultralytics) có nhãn `handgun`, đặt trong
`MyDrive/SENTINEL/datasets/handgun_self/labelled/`. Clip mp4 tự quay (mọi clip đều có người cầm súng) đặt trong
`handgun_self/clips/` để đo độ ổn định ở bước 6. Chỉ dùng súng đồ chơi / mô hình hợp pháp.

**Cách chạy:**
1. *Runtime → Change runtime type*: **L4** là đủ.
2. *Runtime → Run all*, cho phép mount Google Drive.
3. Phiên **tự ngắt** khi xong hoặc khi lỗi (traceback ghi ra `ERROR.txt` trên Drive).

| Bước | Việc làm | Thời gian ước tính (L4) |
|---|---|---|
| 1 | Pool công khai: lần đầu tải (~1.5 GB Zenodo + Granada + Open Images…) rồi cache `pool_v1.tar` lên Drive | 40–80 phút lần đầu, 2–5 phút các lần sau |
| 2 | Ghép dữ liệu tự quay (nếu có), chia train/val/test theo nhóm nguồn | 2–5 phút |
| 3 | Pose → crop quanh tay → gán nhãn từ box súng, ảnh xem trước | 15–40 phút |
| 4 | Fine-tune ResNet50 (ImageNet), cân bằng lớp, dừng sớm theo AP val | 30–90 phút |
| 5 | Đánh giá, chọn ngưỡng (báo nhầm ≤ 3% trên ảnh không có súng của val), đóng gói `handgun_<RUN_TAG>.zip` | ~5 phút |
| 6 | Độ ổn định trên clip tự quay (nếu có) và báo nhầm trên clip Charades test | 5–15 phút |

**Kết quả trên Drive:** `MyDrive/SENTINEL/artifacts/handgun/<RUN_TAG>/`. Cài trên máy bằng
`extensions\env\Scripts\python.exe -m extensions.handgun.install_handgun handgun_<RUN_TAG>.zip`.

**Lưu ý:** ảnh cận cảnh (Sohas, Granada) thường không thấy cả người nên pose không tìm được tay; ở tập train các ảnh
đó cho crop "giả" quanh box súng (trọng số ×0.5), val/test chỉ dùng crop quanh tay thật.
`hand_coverage_of_weapon_images` ở bước 3 cho biết tỉ lệ ảnh có súng mà pose tìm được bàn tay cầm nó.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, tarfile, time, traceback, urllib.request, zipfile
from google.colab import drive

ARCH = 'resnet50'             # resnet18 / resnet34 nhẹ hơn cho laptop yếu
POSE = 'yolo11s-pose.pt'      # model pose dùng để cắt crop và chạy ở runtime
EPOCHS = 25
BATCH = 96
WORKERS = 4                   # crop nhỏ (256 px) nên 4 worker không tốn nhiều RAM
VN_WEIGHT = 4                 # trọng số lấy mẫu crop tự quay (nhóm 'vn' của build_dataset)
PUBLIC_POOL_VERSION = 'v1'    # pool công khai súng ngắn, cache trên Drive sau lần đầu
REFRESH_PUBLIC_POOL = False
AUTO_DISCONNECT = True        # tự ngắt phiên khi xong hoặc khi lỗi
SYNC_DURING_TRAIN = False     # True: chép last.pt lên Drive mỗi SYNC_MINUTES phút để train tiếp nếu bị ngắt
SYNC_MINUTES = 5              # chu kỳ in RAM/GPU (và chép Drive nếu bật)
SOURCE_REVISION = 'c7bdf783a5b963cf03789c98f42a1b63cfaaa39b'
os.environ['WEAPON_CLASS_MAP'] = 'datasets/handgun/class_map.json'  # mọi script dữ liệu đọc lớp súng
RUN_TAG = 'hand_01'
INIT_FROM = None              # train từ ImageNet
LR = 1e-4
HARD_NEGATIVE_WEIGHT = 3.0    # crop âm COCO / Charades (người cầm đồ vật thường)
NEG_CLIPS = 24                # clip Charades test để đo báo nhầm theo thời gian (0 = bỏ qua)
POOL_SOURCES = [              # (nguồn, tham số thêm) cho collect.py; chỉ chạy khi chưa có cache trên Drive
    ('granada', []),
    ('zenodo', []),
    ('simuletic', []),
    ('openimages', ['--oi-per-class', 2000, '--oi-negatives-per-class', 300]),
    ('coco_hold', ['--coco-neg-limit', 3000]),
    ('charades_neg', ['--charades-videos', 600]),
]

assert RUN_TAG and all(c.isalnum() or c in '_-' for c in RUN_TAG)
subprocess.run(['nvidia-smi'], check=True)
drive.mount('/content/drive', timeout_ms=180000)
roots = [p for p in (Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')) if p.is_dir()]
assert roots, 'Không thấy MyDrive sau khi mount Google Drive'
SENTINEL = roots[0] / 'SENTINEL'
VN_DIR = SENTINEL / 'datasets' / 'handgun_self'
PUBLIC_CACHE = SENTINEL / 'datasets' / 'handgun_public' / f'pool_{PUBLIC_POOL_VERSION}.tar'
OUT = SENTINEL / 'artifacts' / 'handgun' / RUN_TAG
OUT.mkdir(parents=True, exist_ok=True)
(VN_DIR / 'labelled').mkdir(parents=True, exist_ok=True)
(VN_DIR / 'clips').mkdir(parents=True, exist_ok=True)
(OUT / 'drive_write_probe.txt').write_text('write OK', encoding='utf-8')
WORK = Path('/content/handgun_work')
SRC, POOL, DATA, CROPS, SCRATCH = WORK / 'src', WORK / 'pool', WORK / 'dataset', WORK / 'crops', WORK / 'scratch'
LOCAL_OUT, LOGS = WORK / 'out', WORK / 'logs'
for path in (WORK, SCRATCH, LOCAL_OUT, LOGS):
    path.mkdir(parents=True, exist_ok=True)
STARTED = time.time()
SYNC_FILES = ('train/last.pt', 'train/best.pt', 'train/history.json', 'train/TRAIN_DONE', 'train/evaluation.json')


def sync_to_drive():
    """Chép checkpoint, kết quả và log từ ổ phiên lên Drive (ghi file tạm rồi đổi tên)."""
    import threading
    with globals().setdefault('_SYNC_LOCK', threading.Lock()):
        items = [(LOCAL_OUT / rel, OUT / rel) for rel in SYNC_FILES]
        items += [(p, OUT / p.name) for p in LOGS.glob('*.log')]
        items += [(p, OUT / p.name) for p in (LOCAL_OUT / 'train').glob('handgun_*.zip')]
        for source, target in items:
            if not source.is_file():
                continue
            if target.is_file() and target.stat().st_size == source.stat().st_size \
                    and target.stat().st_mtime >= source.stat().st_mtime:
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            partial = target.with_name(target.name + '.partial')
            shutil.copy2(source, partial)
            partial.replace(target)


def disconnect(reason):
    print(reason, flush=True)
    try:
        sync_to_drive()
    except Exception as error:
        print('Không chép được lên Drive:', error, flush=True)
    if AUTO_DISCONNECT:
        print('Phiên Colab sẽ tự ngắt sau 20 giây (đặt AUTO_DISCONNECT = False để giữ lại).', flush=True)
        time.sleep(20)
        drive.flush_and_unmount()
        from google.colab import runtime
        runtime.unassign()


def on_error(shell, etype, evalue, tb, tb_offset=None):
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    text = ''.join(traceback.format_exception(etype, evalue, tb))
    (OUT / 'ERROR.txt').write_text(time.strftime('%Y-%m-%d %H:%M:%S ') + text, encoding='utf-8')
    disconnect(f'LỖI, xem {OUT / "ERROR.txt"} và log trên Drive.')


get_ipython().set_custom_exc((Exception,), on_error)
(OUT / 'ERROR.txt').unlink(missing_ok=True)
print('Kết quả sẽ lưu tại:', OUT)
print('Dữ liệu tự quay đọc từ:', VN_DIR)


In [ ]:
# Mã nguồn được ghim theo commit để kết quả tái lập được.
marker = SRC / '.revision'
if not marker.is_file() or marker.read_text().strip() != SOURCE_REVISION:
    shutil.rmtree(SRC, ignore_errors=True)
    archive = WORK / 'source.zip'
    url = 'https://codeload.github.com/TranDuon/anomaly-behavior-detector-system/zip/' + SOURCE_REVISION
    with urllib.request.urlopen(url, timeout=300) as response, archive.open('wb') as output:
        shutil.copyfileobj(response, output)
    with zipfile.ZipFile(archive) as package:
        top = package.namelist()[0].split('/')[0]
        package.extractall(WORK)
    (WORK / top).rename(SRC)
    marker.write_text(SOURCE_REVISION)
print('Mã nguồn:', SOURCE_REVISION)


def is_progress(line):
    return '%|' in line or ('it/s]' in line) or ('s/it]' in line)


def run(args, log_name, cwd=SRC):
    """Chạy lệnh; log đầy đủ ghi ở ổ phiên, notebook chỉ hiện dòng tóm tắt."""
    started = time.time()
    env = {**os.environ, 'TQDM_MININTERVAL': '30'}
    with (LOGS / log_name).open('a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(map(str, args))}\n')
        process = subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        for line in process.stdout:
            log.write(line)
            if not is_progress(line):
                print(line, end='')
        code = process.wait()
    print(f'--- {log_name}: exit {code}, {(time.time() - started) / 60:.1f} phút')
    if code:
        raise RuntimeError(f'{log_name} lỗi (exit {code}), xem {OUT / log_name}')


def py(module, *args, log):
    run([sys.executable, '-u', '-m', module, *args], log)


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.183', 'hf_xet'], 'install.log')
import torch, torchvision, ultralytics
assert torch.cuda.is_available(), 'Chọn GPU (L4/A100) trong Runtime → Change runtime type'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__, '| torchvision', torchvision.__version__,
      '| ultralytics', ultralytics.__version__)
print('Ổ đĩa trống: %.0f GB' % (shutil.disk_usage('/content').free / 1e9))


## Bước 1 · Pool dữ liệu công khai

Nếu `handgun_public/pool_v1.tar` đã có trên Drive thì chỉ giải nén. Chưa có thì tải từng nguồn (nguồn nào lỗi thì bỏ
qua và in cảnh báo) rồi cache lại lên Drive cho các lần sau.


In [ ]:
if PUBLIC_CACHE.is_file() and not REFRESH_PUBLIC_POOL:
    print('Dùng pool công khai đã cache:', PUBLIC_CACHE, '(%.2f GB)' % (PUBLIC_CACHE.stat().st_size / 1e9))
    shutil.rmtree(POOL, ignore_errors=True)
    with tarfile.open(PUBLIC_CACHE) as bundle:
        bundle.extractall(WORK)
else:
    failed = []
    for source, extra in POOL_SOURCES:
        try:
            py('extensions.sharp_object.collect', 'labeled', '--root', POOL, '--work', SCRATCH,
               '--sources', source, *extra, log='collect.log')
        except RuntimeError as error:
            failed.append(source)
            print('CẢNH BÁO: bỏ qua nguồn', source, '-', error)
        shutil.rmtree(SCRATCH, ignore_errors=True)
        SCRATCH.mkdir(parents=True, exist_ok=True)
    assert (POOL / 'manifest.csv').is_file(), 'Không tải được nguồn công khai nào'
    PUBLIC_CACHE.parent.mkdir(parents=True, exist_ok=True)
    partial = PUBLIC_CACHE.with_suffix('.partial')
    with tarfile.open(partial, 'w') as bundle:
        bundle.add(POOL, arcname='pool')
    partial.replace(PUBLIC_CACHE)
    print('Đã cache pool lên Drive:', PUBLIC_CACHE, '| nguồn lỗi:', failed or 'không')
print(json.dumps(json.loads((POOL / 'stats.json').read_text()), indent=1, ensure_ascii=False))


## Bước 2 · Ghép dữ liệu tự quay và chia tập

Mọi zip / thư mục trong `handgun_self/labelled/` (cũ trước, mới sau). Không có thì chỉ dùng dữ liệu công khai.
Nhóm tự quay được chia val/test riêng (`crop_vn_*`, `alert_vn_*` ở bước 5).


In [ ]:
inputs = sorted((p for p in (VN_DIR / 'labelled').iterdir() if p.suffix.lower() == '.zip' or p.is_dir()),
                key=lambda p: p.stat().st_mtime)
print('Đầu vào tự quay:', [p.name for p in inputs] or 'không có')
local_inputs = []
for index, path in enumerate(inputs):
    target = WORK / 'vn_inputs' / f'{index:02d}_{path.name}'
    target.parent.mkdir(parents=True, exist_ok=True)
    if path.is_dir():
        shutil.copytree(path, target, dirs_exist_ok=True)
    else:
        shutil.copy(path, target)
    local_inputs.append(target)
py('extensions.sharp_object.build_dataset', '--pool', POOL, *(['--vn', *local_inputs] if local_inputs else []),
   '--out', DATA, log='build_dataset.log')
shutil.copy(DATA / 'summary.json', OUT / 'dataset_summary.json')
summary = json.loads((DATA / 'summary.json').read_text())
for split in ('train', 'val', 'test'):
    print(split, '| ảnh:', summary['images'][split], '| tự quay:', summary[split]['vn_images'],
          '| ảnh âm:', summary[split]['negatives'])
for warning in summary['warnings']:
    print('CẢNH BÁO:', warning)


## Bước 3 · Pose → crop quanh tay → gán nhãn

- Pose `POSE` chạy ở 960 px (lớn hơn runtime để bắt được người nhỏ trong ảnh công khai), tối đa 8 người mỗi ảnh.
- Mỗi bàn tay thấy được (độ tin cổ tay ≥ 0.3) cho một crop, lưu 256 px.
- `hand_coverage_of_weapon_images`: tỉ lệ ảnh có súng mà pose tìm được bàn tay cầm nó.
- Ảnh xem trước: hàng trên (vạch đỏ) là crop dương, hàng dưới (vạch xanh) là crop âm. Nên nhìn qua để chắc crop đúng chỗ.


In [ ]:
from IPython.display import Image, display

py('extensions.sharp_object.build_hand_crops', '--data', DATA, '--out', CROPS, '--pose', POSE, '--batch', 32,
   log='hand_crops.log')
crops_summary = json.loads((CROPS / 'summary.json').read_text())
shutil.copy(CROPS / 'summary.json', OUT / 'crops_summary.json')
for split, stat in crops_summary['splits'].items():
    print(split, {k: stat.get(k) for k in ('images', 'images_with_people', 'weapon_images',
                                           'hand_coverage_of_weapon_images', 'hand_pos', 'hand_neg',
                                           'pseudo_pos', 'pseudo_neg', 'ambiguous', 'vn_hand_pos')})
for split in ('train', 'val'):
    preview = CROPS / f'preview_{split}.jpg'
    if preview.is_file():
        shutil.copy(preview, OUT / preview.name)
        print('Xem trước', split)
        display(Image(filename=str(preview)))


## Bước 4 · Fine-tune ResNet

- ResNet `ARCH` pre-train ImageNet, một logit "tay cầm súng ngắn"; AdamW, cosine, AMP; tăng cường crop/scale, lật,
  xoay, màu, mờ.
- Lấy mẫu cân bằng hai lớp; crop tự quay ×`VN_WEIGHT`, crop giả ×0.5, crop âm COCO / Charades ×`HARD_NEGATIVE_WEIGHT`.
- Dừng sớm sau 6 epoch không tăng AP trên crop tay thật của val. Bật `SYNC_DURING_TRAIN` để train tiếp được nếu bị ngắt.


In [ ]:
import threading, torch

INIT_LOCAL = WORK / 'init.pt'
if INIT_FROM:
    assert INIT_FROM.is_file(), f'Không thấy checkpoint khởi đầu {INIT_FROM}'
    shutil.copy(INIT_FROM, INIT_LOCAL)
    print('Train tiếp từ:', INIT_FROM)
drive_last, local_last = OUT / 'train' / 'last.pt', LOCAL_OUT / 'train' / 'last.pt'
if drive_last.is_file() and not local_last.is_file():
    print('Chép run dở từ Drive về ổ phiên:', OUT / 'train')
    shutil.copytree(OUT / 'train', LOCAL_OUT / 'train', dirs_exist_ok=True)
stop_sync = threading.Event()


def sync_loop():
    while not stop_sync.wait(SYNC_MINUTES * 60):
        try:
            sync_to_drive()
        except Exception as error:
            print('Chép lên Drive lỗi (sẽ thử lại):', error, flush=True)


def monitor_loop():
    import psutil
    while not stop_sync.wait(SYNC_MINUTES * 60):
        ram = psutil.virtual_memory()
        gpu = torch.cuda.mem_get_info() if torch.cuda.is_available() else (0, 0)
        print(f'[{time.strftime("%H:%M")}] RAM {ram.used / 1e9:.1f}/{ram.total / 1e9:.1f} GB ({ram.percent:.0f}%) | '
              f'ổ trống {shutil.disk_usage("/content").free / 1e9:.0f} GB | '
              f'GPU dùng {(gpu[1] - gpu[0]) / 1e9:.1f}/{gpu[1] / 1e9:.1f} GB', flush=True)


threading.Thread(target=monitor_loop, daemon=True).start()
if SYNC_DURING_TRAIN:
    threading.Thread(target=sync_loop, daemon=True).start()
common = ['--crops', CROPS, '--output', LOCAL_OUT / 'train', '--arch', ARCH, '--batch', BATCH, '--workers', WORKERS]
try:
    py('extensions.sharp_object.train_hand_classifier', '--stage', 'train', *common, '--epochs', EPOCHS,
       '--vn-weight', VN_WEIGHT, '--lr', LR, '--hard-negative-weight', HARD_NEGATIVE_WEIGHT, *(['--init', INIT_LOCAL] if INIT_FROM else []), log='train.log')
finally:
    stop_sync.set()
    sync_to_drive()


## Bước 5 · Đánh giá, chọn ngưỡng, đóng gói

- AP / AUC trên crop tay thật của val và test (toàn bộ và phần tự quay).
- Cảnh báo theo ảnh: ảnh cảnh báo khi một crop tay ≥ ngưỡng. Ngưỡng thấp nhất sao cho báo nhầm trên ảnh **không có súng**
  của val ≤ 3%. `alert_hard_negative_*`: báo nhầm trên ảnh COCO / Charades người cầm đồ vật thường.


In [ ]:
py('extensions.sharp_object.train_hand_classifier', '--stage', 'evaluate', *common, log='evaluate.log')
py('extensions.sharp_object.train_hand_classifier', '--stage', 'export', *common, '--pose', POSE,
   '--run-tag', RUN_TAG, '--name', 'handgun', log='export.log')
sync_to_drive()
report = json.loads((LOCAL_OUT / 'train' / 'evaluation.json').read_text())
fmt = lambda d: {k: (round(v, 3) if isinstance(v, float) else v) for k, v in d.items()}
for key in ('crop_all_val', 'crop_all_test', 'crop_vn_val', 'crop_vn_test'):
    print(f'{key:16s}', fmt(report[key]))
print('\nNgưỡng cảnh báo:', report['alert_threshold'], '-', report['threshold_rule'])
for key in ('alert_val', 'alert_test', 'alert_vn_test', 'alert_hard_negative_val', 'alert_hard_negative_test'):
    if key in report:
        print(f'{key:24s}', fmt(report[key]))
print('Báo nhầm theo crop (test):', report['crop_false_alarm_test'])
print('\nĐường cong ngưỡng (test):')
for row in report['alert_curve']:
    print('  ', row['threshold'], fmt(row['test']))
print('\nFile trên Drive:')
for path in sorted(OUT.rglob('*')):
    if path.is_file() and path.suffix in {'.zip', '.json', '.pt', '.log', '.jpg', '.txt'}:
        print(f'  {str(path.relative_to(OUT)):40s} {path.stat().st_size / 1e6:8.2f} MB')
(OUT / 'DONE.txt').write_text(f'xong {time.strftime("%Y-%m-%d %H:%M:%S")}, {(time.time() - STARTED) / 3600:.1f} giờ',
                              encoding='utf-8')
print('\nTổng thời gian: %.1f giờ' % ((time.time() - STARTED) / 3600))
print('Bước tiếp theo: tải handgun_' + RUN_TAG + '.zip về máy, cài bằng extensions.handgun.install_handgun rồi thử trên camera.')


## Bước 6 · Clip: độ ổn định (tự quay) và báo nhầm (Charades test)

- Clip trong `handgun_self/clips/` (mọi clip có người cầm súng): `alert_share` càng cao càng tốt, `drops` (điểm tụt dưới
  ngưỡng ngay sau một bước trên ngưỡng) càng ít càng tốt.
- `NEG_CLIPS` clip Charades **test** (người cầm chổi / điện thoại / cốc… trong nhà, không dùng train): `alert_share` càng
  thấp càng tốt.


In [ ]:
clips_dir = VN_DIR / 'clips'
package = LOCAL_OUT / 'train' / 'package'
if any(clips_dir.glob('*.mp4')):
    py('extensions.sharp_object.eval_hand_clips', '--package', package, '--clips', clips_dir,
       '--out', LOCAL_OUT / 'train' / 'clips_eval.json', log='clips_eval.log')
    shutil.copy(LOCAL_OUT / 'train' / 'clips_eval.json', OUT / 'clips_eval.json')
    own = json.loads((LOCAL_OUT / 'train' / 'clips_eval.json').read_text())
    print('CLIP TỰ QUAY (càng cao càng tốt) - tỉ lệ bước vượt ngưỡng:', {k: v['alert_share'] for k, v in own['overall'].items()})
else:
    print('Không có clip tự quay trong', clips_dir)
if NEG_CLIPS:
    neg_dir = WORK / 'neg_clips'
    shutil.rmtree(neg_dir, ignore_errors=True)
    py('extensions.sharp_object.collect', 'charades_test', '--root', neg_dir, '--work', SCRATCH, '--limit', NEG_CLIPS,
       log='neg_clips.log')
    py('extensions.sharp_object.eval_hand_clips', '--package', package, '--clips', neg_dir,
       '--out', LOCAL_OUT / 'train' / 'clips_neg_eval.json', log='clips_eval.log')
    shutil.copy(LOCAL_OUT / 'train' / 'clips_neg_eval.json', OUT / 'clips_neg_eval.json')
    neg = json.loads((LOCAL_OUT / 'train' / 'clips_neg_eval.json').read_text())
    print('BẢN ÂM (càng thấp càng tốt) - tỉ lệ bước vượt ngưỡng:', {k: v['alert_share'] for k, v in neg['overall'].items()})
sync_to_drive()


## Tự ngắt phiên

Kết quả đã nằm trên Drive nên ngắt phiên để không tốn compute unit.


In [ ]:
disconnect('XONG. Kết quả ở ' + str(OUT))
